In [ ]:
# === Cell 1: Mount Drive & Define Paths ===

from google.colab import drive
drive.mount('/content/drive')

import os

BASE_DIR = "/content/drive/MyDrive/brain_tumor_project"

DATA_TRAIN_IMG = f"{BASE_DIR}/data/train/images"
DATA_TRAIN_MASK = f"{BASE_DIR}/data/train/masks"

DATA_TEST_IMG = f"{BASE_DIR}/data/test/images"
DATA_TEST_MASK = f"{BASE_DIR}/data/test/masks"

MODEL_DIR = f"{BASE_DIR}/models/rcnn"
OUTPUT_DIR = f"{BASE_DIR}/models/rcnn/output"
NOTEBOOK_DIR = f"{BASE_DIR}/notebooks"

# Create directories if missing
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(NOTEBOOK_DIR, exist_ok=True)

print("RCNN directories ready.")
print("Train images:", DATA_TRAIN_IMG)
print("Test images:", DATA_TEST_IMG)


Mounted at /content/drive
RCNN directories ready.
Train images: /content/drive/MyDrive/brain_tumor_project/data/train/images
Test images: /content/drive/MyDrive/brain_tumor_project/data/test/images


In [ ]:
# === Cell 2: Install Detectron2 ===

!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'


  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.4/86.4 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 44.3 MB/s eta 0:00:00


In [ ]:
# === Cell 3: Imports ===

import os
import cv2
import torch
import numpy as np
from tqdm import tqdm
import json

from detectron2.structures import BoxMode
from detectron2.engine import DefaultTrainer, DefaultPredictor
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.config import get_cfg
from detectron2 import model_zoo


In [ ]:
# === Cell 4 (FAST VERSION): Convert dataset locally for speed ===

import shutil

# 1. Copy train/test images to fast /content storage
!rm -rf /content/brisc_fast
os.makedirs("/content/brisc_fast/train/images", exist_ok=True)
os.makedirs("/content/brisc_fast/train/masks", exist_ok=True)
os.makedirs("/content/brisc_fast/test/images", exist_ok=True)
os.makedirs("/content/brisc_fast/test/masks", exist_ok=True)

# Copy data locally (much faster to process)
print("Copying dataset to local runtime (fast storage)...")
shutil.copytree(DATA_TRAIN_IMG, "/content/brisc_fast/train/images", dirs_exist_ok=True)
shutil.copytree(DATA_TRAIN_MASK, "/content/brisc_fast/train/masks", dirs_exist_ok=True)
shutil.copytree(DATA_TEST_IMG,  "/content/brisc_fast/test/images", dirs_exist_ok=True)
shutil.copytree(DATA_TEST_MASK, "/content/brisc_fast/test/masks", dirs_exist_ok=True)

# Update paths to local copies
LOCAL_TRAIN_IMG = "/content/brisc_fast/train/images"
LOCAL_TRAIN_MASK = "/content/brisc_fast/train/masks"
LOCAL_TEST_IMG  = "/content/brisc_fast/test/images"
LOCAL_TEST_MASK = "/content/brisc_fast/test/masks"

print("Local dataset ready. Starting conversion...")

def create_detectron_dataset(img_dir, mask_dir):
    data = []
    img_files = sorted(os.listdir(img_dir))

    for idx, fname in enumerate(img_files):
        if not fname.endswith(".jpg"):
            continue

        img_path = os.path.join(img_dir, fname)
        mask_path = os.path.join(mask_dir, fname.replace(".jpg", ".png"))

        img = cv2.imread(img_path)
        if img is None:
            continue

        h, w = img.shape[:2]

        mask = cv2.imread(mask_path, 0)
        if mask is None:
            continue

        _, thresh = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        annotations = []
        for c in contours:
            if len(c) < 3:
                continue

            x, y, bw, bh = cv2.boundingRect(c)
            bbox = [x, y, x + bw, y + bh]

            seg = c.flatten().tolist()
            if len(seg) < 6:
                continue

            annotations.append({
                "bbox": bbox,
                "bbox_mode": BoxMode.XYXY_ABS,
                "segmentation": [seg],
                "category_id": 0,
                "iscrowd": 0
            })

        if not annotations:
            continue

        record = {
            "file_name": img_path,
            "height": h,
            "width": w,
            "image_id": idx,
            "annotations": annotations
        }
        data.append(record)

    return data

train_dataset = create_detectron_dataset(LOCAL_TRAIN_IMG, LOCAL_TRAIN_MASK)
test_dataset  = create_detectron_dataset(LOCAL_TEST_IMG, LOCAL_TEST_MASK)

print("Train samples:", len(train_dataset))
print("Test samples:", len(test_dataset))
print("Dataset conversion complete!")


Copying dataset to local runtime (fast storage)...


KeyboardInterrupt: 

In [ ]:
# === Cell 5: Register datasets ===

def register_brisc():
    DatasetCatalog.register("brisc_train", lambda: train_dataset)
    MetadataCatalog.get("brisc_train").set(thing_classes=["tumor"])

    DatasetCatalog.register("brisc_test", lambda: test_dataset)
    MetadataCatalog.get("brisc_test").set(thing_classes=["tumor"])

register_brisc()

metadata = MetadataCatalog.get("brisc_train")
print("Registered datasets:", DatasetCatalog.list())


In [ ]:
# === Cell 6: Create Mask R-CNN config ===

cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file(
    "COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"
))

cfg.DATASETS.TRAIN = ("brisc_train",)
cfg.DATASETS.TEST = ("brisc_test",)

cfg.DATALOADER.NUM_WORKERS = 2
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(
    "COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"
)

cfg.SOLVER.IMS_PER_BATCH = 2
cfg.SOLVER.BASE_LR = 0.00025
cfg.SOLVER.MAX_ITER = 2500

cfg.OUTPUT_DIR = OUTPUT_DIR
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

print("Mask R-CNN configuration created.")


Mask R-CNN configuration created.


In [ ]:
# === Cell 7: Train Mask R-CNN ===

trainer = DefaultTrainer(cfg)
trainer.resume_or_load(resume=False)
trainer.train()


In [ ]:
# === Cell 8: Load trained model ===

cfg.MODEL.WEIGHTS = os.path.join(cfg.OUTPUT_DIR, "model_final.pth")
predictor = DefaultPredictor(cfg)

print("Mask R-CNN ready for inference!")


Mask R-CNN ready for inference!


In [ ]:
# === Cell 9: Dice & IoU ===

def dice(pred, gt):
    pred = pred.astype(bool)
    gt = gt.astype(bool)
    inter = (pred & gt).sum()
    return (2 * inter) / (pred.sum() + gt.sum() + 1e-6)

def iou(pred, gt):
    pred = pred.astype(bool)
    gt = gt.astype(bool)
    inter = (pred & gt).sum()
    union = (pred | gt).sum()
    return inter / (union + 1e-6)


In [ ]:
# === Cell 10: Evaluation ===

dice_scores = []
iou_scores = []

for item in tqdm(test_dataset):
    img = cv2.imread(item["file_name"])
    fname = os.path.basename(item["file_name"])
    mask_path = os.path.join(DATA_TEST_MASK, fname.replace(".jpg", ".png"))
    gt = cv2.imread(mask_path, 0)
    gt_bin = gt > 127

    # Model prediction
    outputs = predictor(img)
    masks = outputs["instances"].pred_masks.cpu().numpy()

    if len(masks) == 0:
        pred_bin = np.zeros_like(gt_bin)
    else:
        pred_bin = np.any(masks, axis=0)

    dice_scores.append(dice(pred_bin, gt_bin))
    iou_scores.append(iou(pred_bin, gt_bin))

print("\n=== MASK R-CNN PERFORMANCE (BRISC TEST) ===")
print("Dice:", sum(dice_scores)/len(dice_scores))
print("IoU: ", sum(iou_scores)/len(iou_scores))
print("============================================\n")


In [ ]:
# === Cell 11: Save evaluation results ===

results_path = f"{MODEL_DIR}/maskrcnn_results.json"

results = {
    "dice": float(sum(dice_scores)/len(dice_scores)),
    "iou": float(sum(iou_scores)/len(iou_scores))
}

with open(results_path, "w") as f:
    json.dump(results, f, indent=4)

print("Saved RCNN results to:", results_path)


In [ ]:
# Install Detectron2 (if not installed)
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'


  Preparing metadata (setup.py) ... done


In [ ]:
%%writefile /content/drive/MyDrive/brain_tumor_project/src/inference_maskrcnn.py
import cv2
import torch
import numpy as np
import gradio as gr

from detectron2.config import get_cfg
from detectron2.engine import DefaultPredictor
from detectron2 import model_zoo

# ================================
# 1. Load Mask R-CNN Model
# ================================

MODEL_PATH = "/content/drive/MyDrive/brain_tumor_project/models/rcnn/output/model_final.pth"

cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file(
    "COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"
))

cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1  # tumor
cfg.MODEL.WEIGHTS = MODEL_PATH
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5
cfg.MODEL.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

predictor = DefaultPredictor(cfg)

# ================================
# 2. Prediction Function
# ================================

def predict_mask(image):
    # Convert to BGR (Detectron2 expects BGR)
    img_bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

    outputs = predictor(img_bgr)

    masks = outputs["instances"].pred_masks.cpu().numpy()
    scores = outputs["instances"].scores.cpu().numpy()

    # If no mask found
    if len(masks) == 0:
        empty = np.zeros_like(image)
        return image, empty, "No tumor detected"

    # Use the union of masks
    combined_mask = np.any(masks, axis=0).astype(np.uint8) * 255

    # Overlay mask on image
    overlay = image.copy()
    overlay[combined_mask > 127] = [255, 0, 0]  # red

    blended = cv2.addWeighted(image, 0.7, overlay, 0.3, 0)

    return blended, combined_mask, f"Tumor detected! (instances: {len(masks)})"

# ================================
# 3. Build Gradio Interface
# ================================

iface = gr.Interface(
    fn=predict_mask,
    inputs=gr.Image(type="numpy", label="Upload MRI Image (JPG)"),
    outputs=[
        gr.Image(type="numpy", label="Overlay Output"),
        gr.Image(type="numpy", label="Predicted Mask"),
        gr.Textbox(label="Status")
    ],
    title="Mask R-CNN Brain Tumor Segmentation",
    description="Upload an MRI slice and detect tumor region using Mask R-CNN."
)

if __name__ == "__main__":
    iface.launch(share=True)


Overwriting /content/drive/MyDrive/brain_tumor_project/src/inference_maskrcnn.py


In [ ]:
!python /content/drive/MyDrive/brain_tumor_project/src/inference_maskrcnn.py


Traceback (most recent call last):
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1331, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 935, in _load_unlocked
  File "<frozen importlib._bootstrap_external>", line 999, in exec_module
  File "<frozen importlib._bootstrap>", line 488, in _call_with_frames_removed
  File "/usr/local/lib/python3.12/dist-packages/gradio/_simple_templates/__init__.py", line 1, in <module>
    from .simpledropdown import SimpleDropdown
  File "/usr/local/lib/python3.12/dist-packages/gradio/_simple_templates/simpledropdown.py", line 7, in <module>
    from gradio.components.base import Component, FormComponent
  File "/usr/local/lib/python3.12/dist-packages/gradio/components/__init__.py", line 31, in <module>
    from gradio.components.gallery import Gallery
  File "/usr/local/lib/python3.12/dist-packages/gradio/components/gallery.py", line 52, in <module>
    class Galle

this is a combine app for both the models to visulize the results


In [ ]:
%cd /content/drive/MyDrive/brain_tumor_project


/content/drive/MyDrive/brain_tumor_project


In [ ]:

%%writefile src/brain_tumor_demo_app.py
import sys, os
sys.path.append("/content/drive/MyDrive/brain_tumor_project/src")

import cv2
import numpy as np
import torch
import gradio as gr

from unet_model import UNet

from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2 import model_zoo

# -----------------------------
# Paths & device
# -----------------------------
UNET_MODEL_PATH = "/content/drive/MyDrive/brain_tumor_project/models/unet_brisc.pth"
RCNN_MODEL_PATH = "/content/drive/MyDrive/brain_tumor_project/models/rcnn/output/model_final.pth"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# -----------------------------
# Load U-Net model
# -----------------------------
unet = UNet().to(DEVICE)
unet.load_state_dict(torch.load(UNET_MODEL_PATH, map_location=DEVICE))
unet.eval()
print("Loaded U-Net weights.")

# -----------------------------
# Load Mask R-CNN (Detectron2)
# -----------------------------
cfg = get_cfg()
cfg.merge_from_file(
    model_zoo.get_config_file(
        "COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"
    )
)

cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1  # single class: tumor
cfg.MODEL.WEIGHTS = RCNN_MODEL_PATH
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5
cfg.INPUT.MIN_SIZE_TEST = 800
cfg.INPUT.MAX_SIZE_TEST = 1333
cfg.MODEL.DEVICE = DEVICE

predictor_rcnn = DefaultPredictor(cfg)
print("Loaded Mask R-CNN weights.")

# -----------------------------
# Helper: U-Net prediction
# -----------------------------
def run_unet(image_rgb: np.ndarray):
    """
    image_rgb: H x W x 3, RGB (from Gradio)
    Returns: overlay RGB, mask uint8, confidence float
    """
    h, w, _ = image_rgb.shape

    # Grayscale + resize to 256x256 (as in training)
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    resized = cv2.resize(gray, (256, 256))
    normalized = resized.astype(np.float32) / 255.0

    tensor = torch.from_numpy(normalized).unsqueeze(0).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        logits = unet(tensor)
        probs = torch.sigmoid(logits)[0, 0]  # (256, 256)

    mask_small = probs.cpu().numpy()
    # confidence: max probability in predicted mask
    confidence = float(mask_small.max())

    # Resize back to original image size
    mask_full = cv2.resize(mask_small, (w, h))
    binary = mask_full > 0.5

    overlay = image_rgb.copy()
    # Paint tumor region in red
    overlay[binary] = [255, 0, 0]
    blended = cv2.addWeighted(image_rgb, 0.7, overlay, 0.3, 0)

    mask_uint8 = (binary.astype(np.uint8) * 255)

    return blended, mask_uint8, confidence

# -----------------------------
# Helper: Mask R-CNN prediction
# -----------------------------
def run_rcnn(image_rgb: np.ndarray):
    """
    image_rgb: H x W x 3, RGB
    Returns: overlay RGB, mask uint8, confidence float
    """
    # Detectron2 expects BGR
    image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)

    outputs = predictor_rcnn(image_bgr)
    instances = outputs["instances"].to("cpu")

    h, w, _ = image_rgb.shape

    if len(instances) == 0:
        empty = np.zeros((h, w), dtype=np.uint8)
        return image_rgb, empty, 0.0

    masks = instances.pred_masks.numpy()  # (N, H, W)
    scores = instances.scores.numpy()

    best_idx = scores.argmax()
    best_mask = masks[best_idx]
    confidence = float(scores[best_idx])

    overlay = image_rgb.copy()
    # Paint tumor region in green
    overlay[best_mask] = [0, 255, 0]
    blended = cv2.addWeighted(image_rgb, 0.7, overlay, 0.3, 0)

    mask_uint8 = (best_mask.astype(np.uint8) * 255)

    return blended, mask_uint8, confidence

# -----------------------------
# Gradio app logic
# -----------------------------
def segment_image(image, model_choice):
    """
    image: RGB numpy array from Gradio
    model_choice: "U-Net", "Mask R-CNN", or "Both"
    """
    if image is None:
        return None, None, "Please upload an MRI image."

    if model_choice == "U-Net":
        overlay, mask, conf = run_unet(image)
        text = f"U-Net confidence (max prob): {conf:.3f}"
        return overlay, mask, text

    elif model_choice == "Mask R-CNN":
        overlay, mask, conf = run_rcnn(image)
        text = f"Mask R-CNN confidence (top detection score): {conf:.3f}"
        return overlay, mask, text

    else:  # Both
        overlay_u, mask_u, conf_u = run_unet(image)
        overlay_r, mask_r, conf_r = run_rcnn(image)

        # Side-by-side overlays and masks
        overlay_combined = np.concatenate([overlay_u, overlay_r], axis=1)
        mask_u_rgb = cv2.cvtColor(mask_u, cv2.COLOR_GRAY2RGB)
        mask_r_rgb = cv2.cvtColor(mask_r, cv2.COLOR_GRAY2RGB)
        mask_combined = np.concatenate([mask_u_rgb, mask_r_rgb], axis=1)

        text = (
            f"U-Net confidence: {conf_u:.3f} | "
            f"Mask R-CNN confidence: {conf_r:.3f}"
        )
        return overlay_combined, mask_combined, text

# -----------------------------
# Build Gradio UI
# -----------------------------
with gr.Blocks(title="Brain Tumor Segmentation: U-Net vs Mask R-CNN") as demo:
    gr.Markdown(
        """
        # Brain Tumor Segmentation Demo
        Upload a T1-weighted brain MRI slice and choose which model to use.

        - **U-Net** → fast pixel-wise segmentation
        - **Mask R-CNN** → instance-level segmentation
        - **Both** → side-by-side comparison
        """
    )

    with gr.Row():
        with gr.Column():
            img_input = gr.Image(
                label="Upload MRI slice (T1)",
                type="numpy"
            )
            model_choice = gr.Radio(
                ["U-Net", "Mask R-CNN", "Both"],
                value="U-Net",
                label="Select Model"
            )
            run_btn = gr.Button("Run Segmentation")

        with gr.Column():
            overlay_out = gr.Image(label="Segmentation Overlay")
            mask_out = gr.Image(label="Predicted Mask")
            conf_out = gr.Label(label="Model Confidence")

    run_btn.click(
        fn=segment_image,
        inputs=[img_input, model_choice],
        outputs=[overlay_out, mask_out, conf_out]
    )

if __name__ == "__main__":
    demo.launch(share=True)


Overwriting src/brain_tumor_demo_app.py


In [ ]:
%cd /content/drive/MyDrive/brain_tumor_project
!python src/brain_tumor_demo_app.py


/content/drive/MyDrive/brain_tumor_project
Using device: cuda
Loaded U-Net weights.
Loaded Mask R-CNN weights.
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://dd606aabb22e4d7dc8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4317.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
W1209 22:34:44.188000 6912 torch/fx/_symbolic_trace.py:52] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for 